# CosMx cell-type validation, visualization, and cross-dataset differential-expression comparison

This notebook reproduces the CosMx components of Supplemental Figure 3 and Figure 4.

Workflow:

1. Load the preprocessed, consensus-labeled CosMx AnnData object.
2. Split neuronal and non-neuronal cells and generate curated-marker dot plots.
3. Export count matrices and metadata for ALDEx analysis.
4. Calculate PCA, neighbors, and UMAP and generate cell-type and QC plots.
5. Postprocess CosMx ALDEx outputs.
6. Compare CosMx and primary MERFISH ALDEx effect sizes using sign concordance, correlations, and CAT@K.
7. Apply the adapted rank-based pseudobulk method to both datasets.
8. Repeat the cross-dataset comparisons for the rank-based results.

The scientific logic and thresholds follow the original analysis notebook. Paths and repeated operations have been centralized for readability and reproducibility.

## 1. Imports and configuration

In [ ]:
from __future__ import annotations

import json
from glob import glob
from pathlib import Path
from typing import Iterable

import anndata as ad
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from scipy.stats import mannwhitneyu

import kimlabspatial.differential_expression as de
import kimlabspatial.preprocessing as pp

from scale_aware_st import (
    RepositoryConfig,
    index_result_tables,
    load_pooled_aldex_spec,
    load_pooled_strata,
)


# ---------------------------------------------------------------------
# Project paths
# ---------------------------------------------------------------------
config = RepositoryConfig.from_env()
ALDEX_RESULT_MODE = "published"  # choose "published" or "recompute"
PROJECT_DIR = config.root
DATA_DIR = config.data_dir
RESULTS_DIR = config.results_dir
FIGURE_DIR = RESULTS_DIR / "figures"
DE_INPUT_DIR = RESULTS_DIR / "aldex_inputs"
DE_RESULT_DIR = RESULTS_DIR / "aldex_results"
TABLE_DIR = RESULTS_DIR / "tables"

COSMX_H5AD = config.data_dir / "cosmx" / "cosmx_data_ct_final.h5ad"
PRIMARY_H5AD = config.data_dir / "primary_merfish" / "analysis_objects" / "adata_glia_aldex_pp.h5ad"

CELL_TYPE_MARKERS_JSON = config.resources_dir / "cosmx" / "cosmx_cell_type_markers.json"
NEURON_MARKERS_JSON = config.resources_dir / "cosmx" / "cosmx_neuron_markers.json"

COSMX_ALDEX_GLOB = str(DE_RESULT_DIR / "cosmx_data_ct_aldex_mem_results_glia" / "cosmx_data_ct_blmm_all_*.xlsx")
COSMX_ALDEX_PREFIX = str(DE_RESULT_DIR / "cosmx_data_ct_aldex_mem_results_glia" / "cosmx_data_ct_blmm_all_")
COSMX_ALDEX_SUFFIX = "_results_05212026"

PRIMARY_ALDEX_ALL = DE_RESULT_DIR / "aldex_main_ds_ALL_ct_results.xlsx"

for directory in (FIGURE_DIR, DE_INPUT_DIR, DE_RESULT_DIR, TABLE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

if ALDEX_RESULT_MODE not in {"published", "recompute"}:
    raise ValueError("ALDEX_RESULT_MODE must be 'published' or 'recompute'.")
required_mode_inputs = (
    [config.additional_data_dir / "Additional_File_4.xlsx", config.additional_data_dir / "Additional_File_6.xlsx"]
    if ALDEX_RESULT_MODE == "published"
    else [COSMX_H5AD, PRIMARY_H5AD]
)
missing_mode_inputs = [path for path in required_mode_inputs if not path.is_file()]
if missing_mode_inputs:
    raise FileNotFoundError(
        f"{ALDEX_RESULT_MODE.title()} mode is missing required input(s): "
        + ", ".join(map(str, missing_mode_inputs))
    )
print(
    "Published mode: using frozen pooled CosMx/primary ALDEx and pseudobulk tables."
    if ALDEX_RESULT_MODE == "published"
    else "Recompute mode is staged: export CosMx inputs, run the separate R workflow, then continue Python postprocessing. See RECOMPUTE_WORKFLOW.md."
)


# ---------------------------------------------------------------------
# Analysis constants retained from the original notebook
# ---------------------------------------------------------------------
CELL_TYPE_KEY = "final_class_final"
COSMX_AGE_KEY = "age"
COSMX_PAIR_KEY = "pair"
MIN_CELLS_FOR_DE = 1_000

VALID_COMPARISON_CELL_TYPES = ["Astro", "EC", "Ependymal", "Immune", "OLG"]
CAT_K_VALUES = list(range(5, 21))

PAIR_COLORS = ["#9467bd", "#8c564b", "#2ca02c", "#d62728"]

## 2. Helper functions

In [ ]:
def load_json(path: Path) -> dict[str, list[str]]:
    """Load a marker dictionary from JSON."""
    if not path.is_file():
        raise FileNotFoundError(f"Marker JSON does not exist: {path}")
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def retain_present_markers(
    marker_dict: dict[str, list[str]],
    var_names: pd.Index,
) -> dict[str, list[str]]:
    """Retain marker genes present in the supplied AnnData object."""
    available = set(var_names.astype(str))
    filtered = {
        group: [gene for gene in genes if gene in available]
        for group, genes in marker_dict.items()
    }
    missing = {
        group: [gene for gene in genes if gene not in available]
        for group, genes in marker_dict.items()
    }
    missing = {group: genes for group, genes in missing.items() if genes}
    if missing:
        print("Markers absent from the CosMx panel or filtered object:")
        for group, genes in missing.items():
            print(f"  {group}: {', '.join(genes)}")
    return {group: genes for group, genes in filtered.items() if genes}


def write_excel_sheets(
    tables: dict[str, pd.DataFrame],
    output_path: Path,
) -> None:
    """Write a dictionary of DataFrames to a multi-sheet Excel workbook."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(output_path) as writer:
        for sheet_name, table in tables.items():
            table.to_excel(writer, sheet_name=sheet_name[:31])


def read_excel_sheets(
    workbook: Path,
    *,
    index_column: str = "gene",
) -> dict[str, pd.DataFrame]:
    """Read all sheets from an Excel workbook and index them by gene."""
    results: dict[str, pd.DataFrame] = {}
    for sheet_name in pd.ExcelFile(workbook).sheet_names:
        table = pd.read_excel(workbook, sheet_name=sheet_name)
        unnamed = [column for column in table.columns if str(column).startswith("Unnamed:")]
        if unnamed:
            table = table.drop(columns=unnamed)
        table = table.set_index(index_column)
        table.index.name = None
        results[sheet_name] = table
    return results


def postprocess_aldex2(
    aldex_results: Iterable[str],
    output_dir: Path,
    filename_prefix: str,
    filename_suffix: str,
    output_stem: str,
    experimental_column: str,
    pivot_filename: str | None,
    *,
    significant: bool = True,
    gene_info=False,
    make_pivot: bool = True,
    p_threshold: float = 0.05,
    signed: bool = False,
) -> dict[str, pd.DataFrame]:
    """
    Postprocess per-cell-type ALDEx result workbooks.

    This preserves the original filtering and signed-pivot logic while using
    Path objects and returning the processed result dictionary.
    """
    aldex_results = list(aldex_results)
    output_dir.mkdir(parents=True, exist_ok=True)

    if gene_info and isinstance(gene_info, (str, pd.DataFrame)):
        gene_info = pp.extract_gene_info(gene_info)

    cell_types = []
    for result_path in aldex_results:
        result_string = str(result_path)
        start = result_string.find(filename_prefix) + len(filename_prefix)
        end = result_string.find(filename_suffix)
        if start < len(filename_prefix) or end < 0:
            raise ValueError(f"Could not parse cell type from filename: {result_path}")
        cell_types.append(result_string[start:end])

    results: dict[str, pd.DataFrame] = {}

    for cell_type, result_path in zip(cell_types, aldex_results):
        print(result_path)
        result_df = pd.read_excel(result_path, engine="openpyxl")

        if significant:
            result_df = result_df[
                result_df[f"{experimental_column}:pval.adj"] <= p_threshold
            ].copy()
        else:
            result_df = result_df.copy()

        if gene_info and isinstance(gene_info, dict):
            descriptions = []
            for gene in result_df["gene"].astype(str):
                descriptions.append(", ".join(gene_info.get(gene.capitalize(), [])))
            gene_position = result_df.columns.get_loc("gene")
            result_df.insert(gene_position + 1, "Gene_info", descriptions)

        results[cell_type] = result_df

    write_excel_sheets(results, output_dir / f"{output_stem}.xlsx")

    if make_pivot:
        combined_genes = pd.concat(
            [table[["gene"]] for table in results.values()],
            ignore_index=True,
        )
        gene_counts = combined_genes.value_counts().reset_index(name="count")
        gene_counts = gene_counts.set_index("gene")
        gene_counts.index.name = None

        for cell_type, table in results.items():
            if signed:
                effect_map = table.set_index("gene")[f"{experimental_column}:est"]
                gene_counts[cell_type] = gene_counts.index.map(
                    lambda gene: (
                        "+"
                        if gene in effect_map.index and effect_map.loc[gene] > 0
                        else "-"
                        if gene in effect_map.index and effect_map.loc[gene] < 0
                        else 0
                    )
                )
            else:
                gene_counts[cell_type] = gene_counts.index.isin(
                    table["gene"]
                ).astype(int)

        if signed:
            cell_type_columns = [
                column for column in gene_counts.columns if column != "count"
            ]
            positive = gene_counts[cell_type_columns].eq("+").sum(axis=1)
            negative = gene_counts[cell_type_columns].eq("-").sum(axis=1)
            insert_position = gene_counts.columns.get_loc("count") + 1
            gene_counts.insert(insert_position, "positive", positive)
            gene_counts.insert(insert_position + 1, "negative", negative)
            totals = gene_counts["positive"] + gene_counts["negative"]
            agreement = (
                gene_counts[["positive", "negative"]].max(axis=1) / totals * 100
            ).fillna(0)
            gene_counts.insert(insert_position + 2, "sign_agreement", agreement)

        if pivot_filename is None:
            raise ValueError("pivot_filename is required when make_pivot=True.")
        gene_counts.to_excel(output_dir / pivot_filename)

    return results


def build_effect_comparisons(
    primary_results: dict[str, pd.DataFrame],
    cosmx_results: dict[str, pd.DataFrame],
    *,
    primary_effect_column: str,
    cosmx_effect_column: str,
) -> tuple[dict[str, pd.DataFrame | None], dict[str, list[str] | None]]:
    """Align shared genes and construct paired effect-size tables."""
    comparisons: dict[str, pd.DataFrame | None] = {
        key: None for key in primary_results
    }
    common_genes: dict[str, list[str] | None] = {
        key: None for key in primary_results
    }

    for key, primary_df in primary_results.items():
        if key not in cosmx_results:
            continue

        cosmx_df = cosmx_results[key]
        shared = sorted(set(primary_df.index) & set(cosmx_df.index))
        common_genes[key] = shared

        comparison = pd.concat(
            [
                primary_df.loc[shared, primary_effect_column],
                cosmx_df.loc[shared, cosmx_effect_column],
            ],
            axis=1,
        )
        comparison.columns = ["effect_ours", "effect_theirs"]
        comparison = comparison.dropna()

        if comparison.shape[0] >= 2:
            comparisons[key] = comparison

    return comparisons, common_genes


def make_sign_matrix(
    comparisons: dict[str, pd.DataFrame | None],
) -> pd.DataFrame:
    """Encode concordant signs as 1, discordant signs as -1, and absent as NaN."""
    valid = {key: value for key, value in comparisons.items() if value is not None}
    genes = sorted(set().union(*(table.index for table in valid.values())))
    matrix = pd.DataFrame(index=genes, columns=list(valid))

    for cell_type, table in valid.items():
        product = np.sign(table["effect_ours"]) * np.sign(table["effect_theirs"])
        agreement = np.where(product > 0, 1, np.where(product < 0, -1, np.nan))
        matrix.loc[table.index, cell_type] = agreement

    return matrix.T


def plot_sign_concordance(
    comparisons: dict[str, pd.DataFrame | None],
    *,
    title: str,
    output_path: Path | None = None,
) -> pd.DataFrame:
    """Plot the cross-dataset sign-concordance heatmap."""
    sign_matrix = make_sign_matrix(comparisons)

    colors = ["red", "lightgrey", "green"]
    bounds = [-1.5, -0.5, 0.5, 1.5]
    cmap = ListedColormap(colors)
    cmap.set_bad(color="lightgrey")
    norm = BoundaryNorm(bounds, cmap.N)

    plt.figure(figsize=(15, 5))
    axis = sns.heatmap(
        sign_matrix.astype(float),
        cmap=cmap,
        norm=norm,
        cbar=False,
        linewidths=0.1,
        linecolor="black",
    )
    axis.set_ylabel("Cell type", fontsize=14)
    axis.set_xlabel("Genes", fontsize=14)
    axis.xaxis.tick_top()
    axis.xaxis.set_label_position("top")
    plt.setp(axis.get_xticklabels(), rotation=90, ha="center", va="bottom")

    legend = [
        Patch(color="green", label="Concordant"),
        Patch(color="red", label="Discordant"),
        Patch(color="lightgrey", label="Absent"),
    ]
    axis.legend(
        handles=legend,
        loc="upper left",
        bbox_to_anchor=(1.05, 1),
        frameon=False,
        fontsize=14,
    )
    plt.title(title)
    plt.tick_params(labelsize=12)
    plt.tight_layout()

    if output_path is not None:
        plt.savefig(output_path, dpi=600, bbox_inches="tight")
    plt.show()
    return sign_matrix


def correlation_table(
    comparisons: dict[str, pd.DataFrame | None],
) -> pd.DataFrame:
    """Calculate Pearson and Spearman effect-size correlations by cell type."""
    records = []

    for cell_type, table in comparisons.items():
        if table is None:
            continue
        table = table.dropna(subset=["effect_ours", "effect_theirs"])
        if len(table) <= 2:
            continue

        records.append(
            {
                "Cell type": cell_type,
                "N genes": int(len(table)),
                "Pearson r": table["effect_ours"].corr(
                    table["effect_theirs"], method="pearson"
                ),
                "Spearman ρ": table["effect_ours"].corr(
                    table["effect_theirs"], method="spearman"
                ),
            }
        )

    result = pd.DataFrame(records).sort_values("Cell type").reset_index(drop=True)
    result["Pearson r"] = result["Pearson r"].round(2)
    result["Spearman ρ"] = result["Spearman ρ"].round(2)
    return result


def calculate_cat_at_k(
    comparisons: dict[str, pd.DataFrame | None],
    k_values: Iterable[int],
    *,
    method: str,
) -> pd.DataFrame:
    """Calculate concordance-at-the-top using absolute effect-size ranks."""
    records = []

    for cell_type, table in comparisons.items():
        if table is None:
            continue

        rank_primary = table["effect_ours"].abs().sort_values(ascending=False)
        rank_cosmx = table["effect_theirs"].abs().sort_values(ascending=False)

        for k in k_values:
            if k > len(table):
                continue
            overlap = len(
                set(rank_primary.head(k).index) & set(rank_cosmx.head(k).index)
            ) / k
            records.append(
                {
                    "celltype": cell_type,
                    "K": k,
                    "CAT@K": overlap,
                    "method": method,
                }
            )

    return pd.DataFrame(records)


def plot_cat_at_k(
    cat_table: pd.DataFrame,
    *,
    title: str,
    output_path: Path | None = None,
) -> None:
    """Plot CAT@K curves by cell type."""
    plt.figure(figsize=(6, 4))
    for cell_type in cat_table["celltype"].unique():
        subset = cat_table[cat_table["celltype"] == cell_type]
        plt.plot(subset["K"], subset["CAT@K"], marker="o", label=cell_type)

    plt.xlabel("K")
    plt.ylabel("CAT@K")
    plt.title(title)
    plt.grid(True)
    plt.legend(
        bbox_to_anchor=(1.05, 1),
        loc="upper left",
        frameon=False,
    )
    plt.tight_layout()

    if output_path is not None:
        plt.savefig(output_path, dpi=600, bbox_inches="tight")
    plt.show()


def run_rank_based_pseudobulk(
    adata: ad.AnnData,
    *,
    cell_type_key: str,
    age_key: str,
    replicate_key: str,
    valid_cell_types: Iterable[str],
    young_label: str = "Yng",
    old_label: str = "Old",
) -> dict[str, pd.DataFrame]:
    """
    Apply the original per-gene replicate-mean Mann–Whitney workflow.

    The expression matrix supplied in ``adata.X`` is used without further
    transformation inside this function.
    """
    results: dict[str, list[dict[str, float | str]]] = {}

    for cell_type in np.unique(adata.obs[cell_type_key]):
        if cell_type not in valid_cell_types:
            continue

        subset = adata[adata.obs[cell_type_key] == cell_type].copy()
        results[cell_type] = []

        for gene in subset.var_names:
            values = subset[:, gene].X
            expression = (
                values.toarray().ravel()
                if hasattr(values, "toarray")
                else np.asarray(values).ravel()
            )

            frame = pd.DataFrame(
                {
                    "exp": expression,
                    "age": subset.obs[age_key].values,
                    "replicate": subset.obs[replicate_key].values,
                }
            )
            pseudobulk = (
                frame.groupby(["replicate", "age"], as_index=False)["exp"].mean()
            )
            pseudobulk = pseudobulk.dropna(subset=["exp"])

            young = pseudobulk.loc[pseudobulk["age"] == young_label, "exp"].values
            old = pseudobulk.loc[pseudobulk["age"] == old_label, "exp"].values

            if len(young) < 2 or len(old) < 2:
                continue

            _, p_value = mannwhitneyu(old, young, alternative="two-sided")
            effect = old.mean() - young.mean()

            results[cell_type].append(
                {
                    "gene": gene,
                    "effect": effect,
                    "pval": p_value,
                }
            )

    return {
        cell_type: pd.DataFrame(records).set_index("gene")
        for cell_type, records in results.items()
        if records
    }

## 3. Load the consensus-labeled CosMx data and marker definitions

In [ ]:
cosmx = sc.read_h5ad(COSMX_H5AD)

required_obs = {CELL_TYPE_KEY, COSMX_AGE_KEY, COSMX_PAIR_KEY, "Area.um2"}
missing_obs = required_obs - set(cosmx.obs.columns)
if missing_obs:
    raise KeyError(f"CosMx object is missing required obs columns: {sorted(missing_obs)}")

required_layers = {"raw_counts", "transformed_counts"}
missing_layers = required_layers - set(cosmx.layers)
if missing_layers:
    raise KeyError(f"CosMx object is missing required layers: {sorted(missing_layers)}")

cell_type_markers = retain_present_markers(
    load_json(CELL_TYPE_MARKERS_JSON),
    cosmx.var_names,
)
neuron_markers = retain_present_markers(
    load_json(NEURON_MARKERS_JSON),
    cosmx.var_names,
)

print(cosmx)
print(cosmx.obs[CELL_TYPE_KEY].value_counts())

## 4. Split neuronal and non-neuronal subsets

In [ ]:
is_neuron = (
    cosmx.obs[CELL_TYPE_KEY].astype(str).str.contains("Glut", na=False)
    | cosmx.obs[CELL_TYPE_KEY].astype(str).str.contains("GABA", na=False)
)

cosmx_neuron = cosmx[is_neuron].copy()
cosmx_glia = cosmx[~is_neuron].copy()

# Exclusions retained from the original notebook.
cosmx_glia = cosmx_glia[cosmx_glia.obs[CELL_TYPE_KEY] != "ABC"].copy()
cosmx = cosmx[cosmx.obs[CELL_TYPE_KEY] != "ABC"].copy()

cosmx_neuron = cosmx_neuron[
    cosmx_neuron.obs[CELL_TYPE_KEY] != "OB-CR Glut"
].copy()
cosmx = cosmx[cosmx.obs[CELL_TYPE_KEY] != "OB-CR Glut"].copy()

neuron_order = [
    "IT-ET Glut",
    "NP-CT-L6b Glut",
    "DG-IMN Glut",
    "MH-LH Glut",
    "TH Glut",
    "GABA",
]
present_neuron_order = [
    label for label in neuron_order
    if label in cosmx_neuron.obs[CELL_TYPE_KEY].astype(str).unique()
]
cosmx_neuron.obs[CELL_TYPE_KEY] = pd.Categorical(
    cosmx_neuron.obs[CELL_TYPE_KEY].astype(str),
    categories=present_neuron_order,
    ordered=True,
)

print(f"Neuronal cells: {cosmx_neuron.n_obs:,}")
print(f"Non-neuronal cells: {cosmx_glia.n_obs:,}")

## 5. Curated-marker dot plots — Supplemental Figure 3

In [ ]:
sc.pl.dotplot(
    cosmx_glia,
    var_names=cell_type_markers,
    groupby=CELL_TYPE_KEY,
    cmap="viridis",
    show=False,
)
plt.savefig(
    FIGURE_DIR / "supp_fig3_cosmx_non_neuronal_marker_dotplot.pdf",
    bbox_inches="tight",
)
plt.show()

In [ ]:
sc.pl.dotplot(
    cosmx_neuron,
    var_names=neuron_markers,
    groupby=CELL_TYPE_KEY,
    cmap="viridis",
    show=False,
)
plt.savefig(
    FIGURE_DIR / "supp_fig3_cosmx_neuronal_marker_dotplot.pdf",
    bbox_inches="tight",
)
plt.show()

## 6. Export count matrices and metadata for ALDEx

In [ ]:
cell_counts = cosmx.obs[CELL_TYPE_KEY].value_counts()
valid_cell_types_for_de = cell_counts[cell_counts > MIN_CELLS_FOR_DE].index

cosmx_de = cosmx[
    cosmx.obs[CELL_TYPE_KEY].isin(valid_cell_types_for_de)
].copy()

print("Cell types retained for DE:")
display(cell_counts.loc[valid_cell_types_for_de].to_frame("n_cells"))

if ALDEX_RESULT_MODE == "recompute":
    de.prep_for_aldex2(
        cosmx_de,
        str(DE_INPUT_DIR),
        "cell_type_all",
        obs_key=CELL_TYPE_KEY,
        count_layer=False,
    )
else:
    print("Published mode: skipping CosMx ALDEx count/metadata export.")


## 7. PCA, neighborhood graph, and UMAP — Figure 4

In [ ]:
cosmx.layers["mapping_raw_counts"] = cosmx.X.copy()
cosmx.X = cosmx.layers["transformed_counts"].copy()

sc.tl.pca(cosmx)
sc.pp.neighbors(cosmx)
sc.tl.umap(cosmx)

sc.pl.umap(
    cosmx,
    color=CELL_TYPE_KEY,
    title="Cell type",
    show=False,
)
plt.savefig(
    FIGURE_DIR / "fig4_cosmx_umap_cell_types.pdf",
    bbox_inches="tight",
)
plt.show()

## 8. QC plots grouped by age and pair — Figure 4

In [ ]:
cosmx.obs["volume"] = cosmx.obs["Area.um2"].copy()
cosmx.obs[COSMX_PAIR_KEY] = pd.Categorical(cosmx.obs[COSMX_PAIR_KEY])
cosmx.uns["pair_colors"] = PAIR_COLORS

pp.qc_plots(
    cosmx,
    groupby=[COSMX_AGE_KEY, COSMX_PAIR_KEY],
    save=False,
)

## 9. Postprocess CosMx ALDEx results

### 9.1 Significant results only

In [ ]:
if ALDEX_RESULT_MODE == "recompute":
    cosmx_aldex_files = sorted(glob(COSMX_ALDEX_GLOB))

    if not cosmx_aldex_files:
        raise FileNotFoundError(
            "No individual CosMx ALDEx workbooks were found. The earlier cell exports "
            "inputs but does not fit ALDEx. Run the CosMx R workflow, place its "
            "workbooks under results/aldex_results/cosmx_data_ct_aldex_mem_results_glia/, "
            "then rerun this cell. See RECOMPUTE_WORKFLOW.md."
        )

    cosmx_aldex_significant = postprocess_aldex2(
        aldex_results=cosmx_aldex_files,
        output_dir=DE_RESULT_DIR,
        filename_prefix=COSMX_ALDEX_PREFIX,
        filename_suffix=COSMX_ALDEX_SUFFIX,
        significant=True,
        output_stem="cosmx_SIG_ct_results_glia",
        experimental_column="age_binary",
        pivot_filename="cosmx_SIG_ct_pivot_glia.xlsx",
        make_pivot=True,
        gene_info=False,
        signed=True,
    )
elif ALDEX_RESULT_MODE == "published":
    cosmx_aldex_all_raw = load_pooled_aldex_spec(config.additional_data_dir, "cosmx_celltype")
    cosmx_aldex_significant = {key: table.loc[table["age_binary:pval.adj"] <= 0.05].copy() for key, table in cosmx_aldex_all_raw.items()}
else:
    raise ValueError("ALDEX_RESULT_MODE must be published or recompute")


### 9.2 All results

In [ ]:
if ALDEX_RESULT_MODE == "recompute":
    cosmx_aldex_all = postprocess_aldex2(
        aldex_results=cosmx_aldex_files,
        output_dir=DE_RESULT_DIR,
        filename_prefix=COSMX_ALDEX_PREFIX,
        filename_suffix=COSMX_ALDEX_SUFFIX,
        significant=False,
        output_stem="cosmx_ALL_ct_results",
        experimental_column="age_binary",
        pivot_filename=None,
        make_pivot=False,
        gene_info=False,
        signed=False,
    )
elif ALDEX_RESULT_MODE == "published":
    cosmx_aldex_all = load_pooled_aldex_spec(config.additional_data_dir, "cosmx_celltype")
else:
    raise ValueError("ALDEX_RESULT_MODE must be published or recompute")


## 10. Cross-dataset ALDEx comparison

In [ ]:
if ALDEX_RESULT_MODE == "published":
    cosmx_aldex_results = index_result_tables(
        load_pooled_aldex_spec(config.additional_data_dir, "cosmx_celltype")
    )
    primary_aldex_results = index_result_tables(
        load_pooled_aldex_spec(config.additional_data_dir, "primary_celltype")
    )
else:
    cosmx_aldex_all_file = DE_RESULT_DIR / "cosmx_ALL_ct_results.xlsx"
    missing_comparison_inputs = [
        path for path in (cosmx_aldex_all_file, PRIMARY_ALDEX_ALL) if not path.is_file()
    ]
    if missing_comparison_inputs:
        raise FileNotFoundError(
            "Recomputed cross-platform comparison requires postprocessed CosMx and "
            "primary ALDEx workbooks from the upstream R/notebook 03 workflows. Missing: "
            + ", ".join(map(str, missing_comparison_inputs))
        )
    cosmx_aldex_results = read_excel_sheets(cosmx_aldex_all_file)
    primary_aldex_results = read_excel_sheets(PRIMARY_ALDEX_ALL)

aldex_comparisons, common_genes_by_cell_type = build_effect_comparisons(
    primary_aldex_results,
    cosmx_aldex_results,
    primary_effect_column="age_binary:est",
    cosmx_effect_column="age_binary:est",
)

aldex_comparisons = {
    key: value
    for key, value in aldex_comparisons.items()
    if value is not None
}
print(f"Cell types compared: {list(aldex_comparisons)}")

### 10.1 Sign concordance — Figure 4

In [ ]:
aldex_sign_matrix = plot_sign_concordance(
    aldex_comparisons,
    title="ALDEx sign concordance: primary MERFISH vs CosMx",
    output_path=FIGURE_DIR / "fig4_aldex_sign_concordance.pdf",
)

### 10.2 Effect-size correlation table — Supplemental Table

In [ ]:
aldex_correlation_table = correlation_table(aldex_comparisons)
display(aldex_correlation_table)

aldex_correlation_table.to_excel(
    TABLE_DIR / "supp_table_aldex_cosmx_effect_correlations.xlsx",
    index=False,
)

### 10.3 CAT@K — Figure 4

In [ ]:
cat_df_aldex = calculate_cat_at_k(
    aldex_comparisons,
    CAT_K_VALUES,
    method="ALDEx",
)
plot_cat_at_k(
    cat_df_aldex,
    title="CAT@K across cell types — ALDEx",
    output_path=FIGURE_DIR / "fig4_aldex_cat_at_k.pdf",
)

## 11. Adapted rank-based pseudobulk analysis

In both datasets, raw counts are normalized to a target sum of 500 and log-transformed before replicate-level means are calculated.

The per-gene test remains a two-sided Mann–Whitney U test comparing replicate-level means.

In [ ]:
if ALDEX_RESULT_MODE == "published":
    primary_rank_results = index_result_tables(
        load_pooled_strata(
            config.additional_data_dir / "Additional_File_4.xlsx",
            sheet_name="All_celltype_pseudobulk",
            stratum_column="Celltype",
            gene_column="gene",
        )
    )
    cosmx_rank_results = index_result_tables(
        load_pooled_strata(
            config.additional_data_dir / "Additional_File_6.xlsx",
            sheet_name="Cosmx_celltype_pseudobulk",
            stratum_column="Celltype",
            gene_column="gene",
        )
    )
    print(
        f"Loaded deposited pseudobulk results: {len(primary_rank_results)} primary "
        f"and {len(cosmx_rank_results)} CosMx cell types."
    )
elif ALDEX_RESULT_MODE == "recompute":
    primary = sc.read_h5ad(PRIMARY_H5AD)
    primary.X = primary.layers["raw_counts"].copy()
    
    cosmx_rank = cosmx.copy()
    
    # Restore the original raw counts first
    cosmx_rank.X = cosmx_rank.layers["raw_counts"].copy()
    
    # Then reproduce the original preprocessing used for the rank-based method
    sc.pp.normalize_total(cosmx_rank, target_sum=500)
    sc.pp.log1p(cosmx_rank)
    
    cosmx_rank_results = run_rank_based_pseudobulk(
        cosmx_rank,
        cell_type_key=CELL_TYPE_KEY,
        age_key=COSMX_AGE_KEY,
        replicate_key=COSMX_PAIR_KEY,
        valid_cell_types=VALID_COMPARISON_CELL_TYPES,
    )
    
    # Primary MERFISH expression handling retained from the original notebook.
    primary_rank = primary.copy()
    sc.pp.normalize_total(primary_rank, target_sum=500)
    sc.pp.log1p(primary_rank)
    
    primary_rank_results = run_rank_based_pseudobulk(
        primary_rank,
        cell_type_key="cell_type",
        age_key="Age",
        replicate_key="sample",
        valid_cell_types=VALID_COMPARISON_CELL_TYPES,
    )
else:
    raise ValueError("ALDEX_RESULT_MODE must be 'published' or 'recompute'.")


### 11.1 Save rank-based results

In [ ]:
write_excel_sheets(
    primary_rank_results,
    TABLE_DIR / "maindata_all_ct_pseudobulk.xlsx",
)
write_excel_sheets(
    cosmx_rank_results,
    TABLE_DIR / "cosmx_all_ct_pseudobulk.xlsx",
)

## 12. Cross-dataset rank-based comparison

In [ ]:
rank_comparisons: dict[str, pd.DataFrame] = {}

for cell_type, primary_table in primary_rank_results.items():
    if cell_type not in cosmx_rank_results:
        continue

    cosmx_table = cosmx_rank_results[cell_type]

    # Preserve the ALDEx-derived common-gene universe used in the original notebook.
    common_gene_key = cell_type.lower()
    shared = common_genes_by_cell_type.get(common_gene_key)

    # Fall back to exact key matching if workbook sheet names were not lowercase.
    if shared is None:
        shared = common_genes_by_cell_type.get(cell_type)

    if shared is None:
        raise KeyError(
            f"No ALDEx common-gene list was found for cell type: {cell_type}"
        )

    shared = [
        gene for gene in shared
        if gene in primary_table.index and gene in cosmx_table.index
    ]

    comparison = pd.concat(
        [
            primary_table.loc[shared, "effect"],
            cosmx_table.loc[shared, "effect"],
        ],
        axis=1,
    )
    comparison.columns = ["effect_ours", "effect_theirs"]
    comparison = comparison.dropna()

    if comparison.shape[0] >= 2:
        rank_comparisons[cell_type] = comparison

print(f"Cell types compared: {list(rank_comparisons)}")

### 12.1 Sign concordance — Figure 4

In [ ]:
rank_sign_matrix = plot_sign_concordance(
    rank_comparisons,
    title="Rank-based sign concordance: primary MERFISH vs CosMx",
    output_path=FIGURE_DIR / "fig4_rank_based_sign_concordance.pdf",
)

### 12.2 Effect-size correlation table — Supplemental Table

In [ ]:
rank_correlation_table = correlation_table(rank_comparisons)
display(rank_correlation_table)

rank_correlation_table.to_excel(
    TABLE_DIR / "supp_table_rank_based_cosmx_effect_correlations.xlsx",
    index=False,
)

### 12.3 CAT@K — Figure 4

In [ ]:
cat_df_rank = calculate_cat_at_k(
    rank_comparisons,
    CAT_K_VALUES,
    method="Rank-based",
)
plot_cat_at_k(
    cat_df_rank,
    title="CAT@K across cell types — rank-based method",
    output_path=FIGURE_DIR / "fig4_rank_based_cat_at_k.pdf",
)

## 13. Combined correlation tables

In [ ]:
combined_correlations = pd.concat(
    [
        aldex_correlation_table.assign(Method="ALDEx"),
        rank_correlation_table.assign(Method="Rank-based"),
    ],
    ignore_index=True,
)
combined_correlations = combined_correlations[
    ["Method", "Cell type", "N genes", "Pearson r", "Spearman ρ"]
]
combined_correlations.to_excel(
    TABLE_DIR / "supp_table_cosmx_cross_dataset_correlations.xlsx",
    index=False,
)

display(combined_correlations)

### 13.1 Supplementary Table S4

In [ ]:
corr_table = combined_correlations.copy()

fig, ax = plt.subplots(figsize=(14, 2))
ax.axis("off")

tbl = ax.table(
    cellText=corr_table.values,
    colLabels=corr_table.columns,
    loc="center",
    cellLoc="center",
    colWidths=[0.24, 0.10, 0.16, 0.17, 0.16, 0.17],
)

tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1.0, 1.4)

for (row, col), cell in tbl.get_celld().items():
    cell.set_linewidth(0.4)

    if row == 0:
        cell.set_text_props(weight="bold")
        cell.set_height(cell.get_height() * 1.35)

plt.title(
    "Supplementary Table S4. SenNet CosMx dataset effect-size concordance metrics",
    pad=12,
)

plt.savefig(
    TABLE_DIR / "Supplementary_Table_S4.pdf",
    dpi=300,
    bbox_inches="tight",
)

plt.show()